# N003 · 分支、循环与枚举

**目标：** 将自然语言条件翻译为可终止的循环。

**先修：** N002。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** if/elif；for/while；range；break/continue；循环边界。

**配套讲解来源：** [同名逐章意见](../../comment/003_branching_loops.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章解决"把人话条件翻译成机器循环"的问题，载体是经典 FizzBuzz 和一个小小的偶数求和。

**问题 1（FizzBuzz）**：从 1 数到 n，对每个数 i 按规则输出一个字符串：如果 i 是 3 的倍数，输出 `'Fizz'`；如果 i 是 5 的倍数，输出 `'Buzz'`；如果 i 同时是 3 和 5 的倍数（也就是 15 的倍数），输出 `'FizzBuzz'`；都不是就把 i 自己变成字符串输出。具体例子：输入 `n=3`，输出列表 `['1', '2', 'Fizz']`——因为 1、2 都不整除，转成字符串 `'1'`、`'2'`，而 3 是 3 的倍数所以是 `'Fizz'`。最容易错的地方在 15：15 既是 3 的倍数又是 5 的倍数，必须输出 `'FizzBuzz'`，如果你先判断"是 3 的倍数"就提前返回 `'Fizz'`，就错了一半。

这道题的本质是"条件有交集时的优先级问题"：三个条件不是平行的，而是有包含关系（15 的倍数 ⊂ 3 的倍数），翻译成 if/elif 时必须让小集合先被认领。

**问题 2（偶数求和）**：算从 1 到 n 之间所有偶数的和。具体例子：输入 `n=8`，输出 `20`，因为 1..8 里的偶数是 2、4、6、8，加起来 2+4+6+8=20。如果输入 `n=7`，输出还是 `20`——7 本身是奇数不参与求和，偶数集合没变。

这一章真正要练的是：循环边界（range 到底含不含 n）、分支顺序（严格条件先判）、以及循环保证会停。这三件事合起来，就是把一段自然语言规则（"从 1 到 n 逐个看，逢三整除输出 Fizz……"）翻译成一个**每次迭代都向前推进、并且必然停下来**的循环。

## 2. 基础知识：先读懂这些词

（本节概念按出场顺序排布，遇到哪个查哪个。）

- **if/elif/else 分支**：Python 从上往下逐个检查条件，命中哪个就执行哪一段，执行完跳过其余全部分支。`elif` 是"否则如果"的缩写。关键性质：一次只有一个分支会执行，先到先得——所以条件顺序就是优先级。
- **for 循环**：把一段操作对序列里的每个元素重复执行一遍。`for i in range(1, n+1)` 就是让 i 依次取 1、2、……、n。
- **range（左闭右开区间）**：`range(a, b)` 从 a 数到 b-1 为止，**不包含 b**。想包含 n 就得写 `range(1, n+1)`；`range(2, n+1, 2)` 的第三个参数 2 表示步长，即从 2 开始每次跳 2：2、4、6……
- **while 循环**：只要条件为真就一直重复。它和 for 可以互相改写，for 适合"知道次数"，while 适合"知道停止条件"。
- **break / continue**：`break` 是立刻整个跳出循环；`continue` 是跳过本轮剩下的语句直接进入下一轮。本章代码没用到，但它们是循环控制的常用件。
- **取模判整除**：`i % 3 == 0` 读作"i 除以 3 余 0"，即"i 是 3 的倍数"。这是把自然语言条件翻译成代码的最基本句式。
- **互斥分支**：if/elif/else 的结构保证每个 i 只会落进一个分支，四路分支正好对应四种互不重叠的情况，每个整数都有归属。
- **枚举/线性扫描**：老老实实从第一个看到最后一个。原文说"枚举并不低级"——当题目要求你为 1..n 每个数都产出一个答案时，任何算法至少要写 n 个输出，所以一个一个过一遍已经是最优量级了。
- **可终止**：循环有明确的结束点（i 数到 n 就停），不会永远转下去。
- **累积器（accumulator）**：跨越多轮循环、不断积累结果的变量。本章有两个：`answer`（从空列表开始往里 append 字符串）和 `total`（从 0 开始累加偶数）。累积器的初始化值很讲究：求和用 0、拼列表用空表，保证"加进来不影响已有结果"。
- **append**：列表的方法，把一个元素加到列表尾部（原地修改）。循环 + append 是构建结果列表的标准范式。

## 3. 思路推导：从小例子开始

**思路 A：FizzBuzz。**

- **Step 1：拆条件。** 每个 i 有四种可能：15 的倍数、只是 3 的倍数、只是 5 的倍数、都不是。注意前一种和后两种有重叠——15 的倍数同时也是 3 的倍数。
- **Step 2：严格条件先判。** 因为 if/elif 是先到先得，必须把"最严格的交集条件"（`i % 15 == 0`）放在最前面，否则它会被更宽的 `i % 3 == 0` 提前截走，永远轮不到输出 `'FizzBuzz'`。
- **Step 2.5：检查有没有漏网之鱼。** 四个分支合起来要能接住所有整数：15 的倍数、纯 3 的倍数、纯 5 的倍数、其余——任何 i 必然落在且只落在其中一类。确认这一点后，循环体就完备了。
- **Step 3：循环逐个处理。** 用 `range(1, n+1)` 让 i 从 1 走到 n，每轮把一个结果 `append` 到答案列表尾部。
- **Step 4：数字转字符串。** 都不整除时用 `str(i)` 把整数变成字符串，保证列表里所有元素类型一致。

**手算演示一轮（n=5）**：i=1，`1%15`、`1%3`、`1%5` 都不是 0，走 else，追加 `'1'`；i=2 同理追加 `'2'`；i=3，`3%15=3` 不为 0，但 `3%3=0` 命中第二个分支，追加 `'Fizz'`；i=4 追加 `'4'`；i=5，前两个不中、`5%5=0` 命中第三分支，追加 `'Buzz'`。最终列表 `['1','2','Fizz','4','Buzz']`。**关键一轮 i=15**：`15%15=0` 第一个分支直接命中，追加 `'FizzBuzz'`——如果这行不在最前，15 会先被 `15%3==0` 截走，错误输出 `'Fizz'`。

**思路 B：偶数求和。** 从 2 开始、步长 2、走到 n（`range(2, n+1, 2)` 恰好枚举 2、4、6……不超过 n 的所有偶数），用一个累积变量 `total` 每轮加上当前偶数。手算 n=8：total 从 0 开始，依次加 2 得 2、加 4 得 6、加 6 得 12、加 8 得 20，循环结束返回 20。每轮之后可以打一个快照："处理完前 k 个偶数，total = 前 k 个偶数之和"，例如加完 6 之后 total=12=2+4+6，快照成立——这个"半途对账"的习惯就是第五节正确性论证的雏形。

**小实例表格（“运行示例”部分）**：“运行示例”部分 对 `fizz_buzz(15)` 的结果画了张表，每行是 `[i, i%3==0, i%5==0, 输出]`。挑几行看：i=3 那行是 `[3, True, False, 'Fizz']`；i=5 是 `[5, False, True, 'Buzz']`；i=15 是 `[15, True, True, 'FizzBuzz']`——正是"两个 True 同时出现时输出必须是 FizzBuzz"的那一行，这张表把分支顺序的重要性直接摆在了眼前。整张表共 15 行，两个布尔列的四种组合全部出现（双 False 交给 else、单 True 两行、双 True 一行），你可以自己数一遍：i=1,2,4,7,8,11,13,14 是双 False，i=3,6,9,12 只有 3 整除，i=5,10 只有 5 整除，i=15 双 True。一张表覆盖全部分支组合，这就是"小实例选得巧"的价值。

## 4. 核心代码：fizz_buzz

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def fizz_buzz(n: int) -> list[str]:
    answer = []
    for i in range(1, n + 1):
        if i % 15 == 0:
            answer.append('FizzBuzz')
        elif i % 3 == 0:
            answer.append('Fizz')
        elif i % 5 == 0:
            answer.append('Buzz')
        else:
            answer.append(str(i))
    return answer
```

### 逐段读懂代码

**核心实现（“分段实现”部分）对应实现：**

（对应程序见下方分段实现与完整代码。）

**函数一：`fizz_buzz`**

- `answer = []`：先准备一个空列表当累积容器。开跑之前它是空的，对应"还没处理任何数"的初始状态。
- `for i in range(1,n+1):`：让 i 依次取 1 到 n。注意 `n+1`——range 不含右端点，多加 1 才能覆盖到 n 本身。这是本题最常见的边界错误点：写成 `range(1,n)` 会漏掉最后一个数。
- `if i % 15 == 0:` / `answer.append('FizzBuzz')`：最严格的条件放在最前。15 是 3 和 5 的最小公倍数，`i % 15 == 0` 等价于"既整除 3 又整除 5"。命中就把字符串追加到列表尾部。
- `elif i % 3 == 0:` / `answer.append('Fizz')`：走到这里说明 i 一定不是 15 的倍数，再判断是否只是 3 的倍数。
- `elif i % 5 == 0:` / `answer.append('Buzz')`：同理判断是否只是 5 的倍数。
- `else:` / `answer.append(str(i))`：前面全不中，说明 i 既不是 3 也不是 5 的倍数，用 `str()` 把整数转成字符串（比如 7 变成 `'7'`），保证返回列表是纯字符串列表。
- `return answer`：循环跑完（i 已经数完 1..n）才走到这行，把攒满 n 个元素的列表交出去。

**函数二：`sum_even`**

- `total = 0`：累积器从 0 起步。求和问题的标准初始化——0 加任何数都不影响结果。
- `for i in range(2,n+1,2):`：三个参数分别是起点 2、终点 n+1（不含）、步长 2，于是 i 取 2、4、6……一路跳过所有奇数，最后一个不超过 n 的偶数也覆盖到。这行展示了用步长直接枚举偶数，省去循环内再判断 `i%2==0` 的功夫。
- `total += i`：每轮把当前偶数累加进 total。n=8 时这行会执行 4 次，分别加 2、4、6、8。
- `return total`：循环自然结束后返回累加结果。

**两个常见错写，顺手对照一下**：

- 把 `range(1,n+1)` 写成 `range(1,n)`：结果永远少最后一个元素。n=15 时你会漏掉唯一的 `'FizzBuzz'`，n=8 的偶数求和会漏掉 8 得 12——两个函数各有断言能抓住它（分别是 `fizz_buzz(15)[-1]` 和 `sum_even(8) == 20`）；
- 把 `str(i)` 忘了写、直接 `answer.append(i)`：列表混进整数，`fizz_buzz(3) == ['1','2','Fizz']` 里 `'1' == 1` 不成立，第一条断言立刻挂。Python 不会替你做类型检查，"列表里元素类型一致"要靠自己守。

## 5. 分段实现：按顺序运行

**本章接口：** `fizz_buzz(n)`、`sum_even(n)`

### fizz_buzz

In [1]:
def fizz_buzz(n: int) -> list[str]:
    answer = []
    for i in range(1, n + 1):
        if i % 15 == 0:
            answer.append('FizzBuzz')
        elif i % 3 == 0:
            answer.append('Fizz')
        elif i % 5 == 0:
            answer.append('Buzz')
        else:
            answer.append(str(i))
    return answer

### sum_even

In [2]:
def sum_even(n: int) -> int:
    total = 0
    for i in range(2, n + 1, 2):
        total += i
    return total

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [3]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

show_table(['i', '3整除', '5整除', '输出'], [(i,i%3==0,i%5==0,x) for i,x in enumerate(fizz_buzz(15),1)])

i,3整除,5整除,输出
1,False,False,1
2,False,False,2
3,True,False,Fizz
4,False,False,4
5,False,True,Buzz
6,True,False,Fizz
7,False,False,7
8,False,False,8
9,True,False,Fizz
10,False,True,Buzz


## 7. 正确性、适用条件与复杂度

循环第 i 次结束后，答案恰好包含 1…i 的正确输出。初始化为空；互斥分支覆盖全部整数；终止后得到完整答案。

**代价：** 按整数转字符串代价细计，输出总字符数 O(n log n)；忽略位数时扫描 O(n)，额外辅助空间 O(1)。

### 展开理解

**FizzBuzz 为什么对：** 我们用一个"半途快照"的角度想：循环处理完 i 之后，答案列表里恰好是 1..i 每个数的正确输出，一个不多一个不少。一开始（i 还没开始）列表是空的，对应"1..0 没有数要处理"，成立。往后每推一步：处理第 i 个数时，四个分支把所有整数分成四类——既是 3 又是 5 的倍数（被 `%15` 抓住）、只是 3 的倍数、只是 5 的倍数、两者都不是——任何整数必然落在其中一类且只落一类（因为 if/elif 只执行第一个命中的分支），所以每轮追加的一定是这个 i 的正确答案。一轮一轮推到 i=n，列表就恰好是完整的正确答案，循环也自然停止。另外，分支顺序保证了 15 的倍数优先被更严格的条件认领，这正是条件重叠时的正确处理方式。

**sum_even 为什么对：** range(2, n+1, 2) 枚举的正是"不超过 n 的全部正偶数，从小到大各一次"，total 从 0 起步逐个累加，加完所有偶数后自然得到它们的总和。没有遗漏（所有偶数都在 range 里）、没有重复（每个只出现一次），所以正确。

**复杂度是多少：** 两个函数都要对 1..n 每个数各处理一次，这是 O(n) 次操作。拿具体数字感受：n=10⁵ 时大约十万次循环，现代计算机一眨眼（毫秒级）就算完。输出占的空间是另一回事：FizzBuzz 要返回 n 个字符串，这 O(n) 的空间是题目要求输出的下界，省不掉；额外辅助变量（answer 引用之外只有 i 和 total）只有常数个，所以额外空间 O(1)。如果细究每个数字转成字符串的长度（数字越大字符串越长，平均位数随 n 的位数增长），总输出字符量是 O(n log n)，但通常按"忽略位数"的说法记 O(n) 即可。最后引用原文一句重要观点：当题目要求 1..n 每个数都产出一个输出时，你至少得花 n 次操作去产出它们，所以线性扫描已经匹配了输出的下界，别觉得枚举"低级"。

**一个失效条件（对应 “掌握检查”部分 的自测要求）：** 把约束从"n 是正整数"放宽成"n 可以是 0 或负数"会怎样？`fizz_buzz(0)` 靠 `range(1,1)` 为空安全返回 `[]`（实现仍成立）；`fizz_buzz(-3)` 同理返回 `[]`。但如果题目语义是"负数输入视为非法"，那这两个安静返回空列表的行为就成了"吞错误"——要不要加 `raise ValueError` 就变成一个新的契约决定。结论：边界行为对不对，取决于题目把边界定义成什么。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 一共五行断言：

```python
assert fizz_buzz(3) == ['1', '2', 'Fizz']

assert fizz_buzz(15)[-1] == 'FizzBuzz'

assert fizz_buzz(1) == ['1']

assert sum_even(8) == 20

assert fizz_buzz(0) == []

print('N003: 所有本章断言通过')
```

- `assert fizz_buzz(3) == ['1','2','Fizz']`：正常小值整表测试。n=3 覆盖了 else 分支（1、2 转字符串）和 Fizz 分支（3），并确认返回的是完整列表而不只是最后一个元素。它还顺便验证了元素是字符串 `'1'` 而非整数 1。
- `assert fizz_buzz(15)[-1] == 'FizzBuzz'`：关键特殊值测试。`[-1]` 取列表最后一个元素，对应 i=15。这条专门盯防"分支顺序写反"的 bug：如果你把 `%3` 判断放在 `%15` 前面，15 会输出 `'Fizz'`，这条立刻失败。
- `assert fizz_buzz(1) == ['1']`：最小正常输入边界。n=1 时只有一个元素 `'1'`，验证循环下界 `range(1, ...)` 的起点没写错。
- `assert sum_even(8) == 20`：正常值。手算 2+4+6+8=20，同时验证了步长 2 的枚举和累积逻辑。注意 n=8 是偶数，正好被包含进求和（如果 range 终点写成 `n` 而不是 `n+1`，会漏掉 8 得 12，这条就会失败）。
- `assert fizz_buzz(0) == []`：零输入边界。n=0 时 `range(1,1)` 是空的，一次循环都不跑，应该安静地返回空列表，不报错。这是"循环边界"知识点的守门测试。

**故障推演：哪种写错会挂哪条？**

- 分支顺序写反（`%3` 判在 `%15` 前）：只有 `fizz_buzz(15)[-1]` 挂，其他四条全绿——这是"只有一条测试在报警"时最值得先查分支顺序的原因；
- range 终点漏写 +1：`fizz_buzz(3)` 变成 `['1','2']` ≠ 期望，第一条挂；`fizz_buzz(15)[-1]` 变成 `'14'` 也挂；
- 忘了 `str(i)`：`fizz_buzz(3) == ['1','2','Fizz']` 因 `'1'` 与 1 不相等而挂；
- sum_even 里步长写成 1（`range(2,n+1)`）且靠 if 判断奇偶又忘了写：把奇数也加进去，n=8 会得 36 ≠ 20。

**断言分层小结**：`fizz_buzz(3)` 与 `sum_even(8)` 管正常路径，`fizz_buzz(15)[-1]` 管分支重叠的特殊值，`fizz_buzz(1)` 与 `fizz_buzz(0)` 管两端边界（最小输入与零输入）。五条断言把"正常 / 特殊 / 边界"三层占齐了。

**断言分层一览表**：

| 断言 | 层次 | 它守护的契约 |
| --- | --- | --- |
| `fizz_buzz(3) == ['1','2','Fizz']` | 正常值 | 整表内容、元素为字符串 |
| `fizz_buzz(15)[-1] == 'FizzBuzz'` | 特殊值 | 分支顺序（15 不被 %3 截走） |
| `fizz_buzz(1) == ['1']` | 下边界 | 循环从 1 开始 |
| `sum_even(8) == 20` | 正常值 | 步长枚举 + 累积正确 |
| `fizz_buzz(0) == []` | 零边界 | 空输入安静返回空表 |

In [4]:
assert fizz_buzz(3) == ['1', '2', 'Fizz']

assert fizz_buzz(15)[-1] == 'FizzBuzz'

assert fizz_buzz(1) == ['1']

assert sum_even(8) == 20

assert fizz_buzz(0) == []

print('N003: 所有本章断言通过')

N003: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 不使用列表推导式实现。

**练习 2：** 将for版本改写成while版本。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（不使用列表推导式实现）**：提示——本章 “分段实现”部分 已经就是普通 for 循环版本了，这条练习的意图是让你别用 `['Fizz' if ... else str(i) for i in range(1,n+1)]` 这种一行推导式，而是完整写出"建空列表 → 循环 → append → return"四步，体会累积器的构建过程。手算示例可以用 n=5，期望 `['1','2','Fizz','4','Buzz']`；边界可用 n=1（单元素）和 n=0（空列表）。
- **练习 2（把 for 版本改写成 while 版本）**：提示——for 版本里隐藏着一个计数器 i，while 版本要你自己管理它：初始化 `i = 1`，条件写 `while i <= n:`，循环体末尾别忘了 `i += 1`，否则循环永远不终止。逐行对照两个版本，确认 i 的取值序列完全相同（都从 1 取到 n）。sum_even 同理：起点 2、步长 2，条件 `while i <= n`。验证时直接断言 while 版和 for 版结果相等即可，边界别忘了 n=0（循环体一次不执行）。做完可以再自问一句：`while i < n` 和 `while i <= n` 差在哪？答案正是 range 终点那 +1 的另一种化身——边界错误会换着马甲反复出现。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [5]:
# 完整实现：本单元格不依赖其他单元格。
def fizz_buzz(n: int) -> list[str]:
    answer = []
    for i in range(1, n + 1):
        if i % 15 == 0:
            answer.append('FizzBuzz')
        elif i % 3 == 0:
            answer.append('Fizz')
        elif i % 5 == 0:
            answer.append('Buzz')
        else:
            answer.append(str(i))
    return answer

def sum_even(n: int) -> int:
    total = 0
    for i in range(2, n + 1, 2):
        total += i
    return total

# 示例与回归测试
assert fizz_buzz(3) == ['1', '2', 'Fizz']

assert fizz_buzz(15)[-1] == 'FizzBuzz'

assert fizz_buzz(1) == ['1']

assert sum_even(8) == 20

assert fizz_buzz(0) == []

print('N003: 所有本章断言通过')

N003: 所有本章断言通过


## 11. 代表题与迁移

- **412. Fizz Buzz**：就是本章 `fizz_buzz` 的原题。它练的核心点有两个：一是 if/elif 分支顺序（15 的倍数必须最先判断），二是 range 左闭右开的边界（`n+1`），这两点正是 “自动测试”部分 里 `fizz_buzz(15)[-1]` 和 `fizz_buzz(0)` 两条断言守护的对象。这题也是面试里的常客——它过滤的不是智力，而是你写循环时是否习惯性地检查边界和条件顺序这两件事。

**提交前的自检三问**：第一，我的循环会不会漏掉 i=n（range 终点是不是 n+1）？第二，遇到"同时满足多条件"的数，我最先判断的是不是最严格的那个条件？第三，返回列表里是不是清一色的字符串？三问都过，这题稳。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。